# M13 · Multimodal e fronteira do estado da arte: prática

1. Ler uma fatura digitalizada com um modelo de visão, em texto livre e com esquema.
2. Validar com código e fazer uma segunda leitura.
3. Voz: sintetizar, transcrever com Whisper e medir a WER.
4. Raciocínio: esforço e self-consistency, com custos.
5. Um modelo pequeno no teu computador.
6. O pipeline completo e a sua avaliação.
7. AI Act: classificar os teus projetos.

**Requisitos:** `uv sync --extra multimodal`, `ANTHROPIC_API_KEY`. GPU recomendada para o Whisper large (em CPU usa `openai/whisper-small`). Ollama opcional para a secção 5.

Lição de apoio: [licao.html](licao.html)

In [ ]:
import base64, json, os, subprocess, sys, time
from collections import Counter
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Image as Mostrar, Audio, display
from anthropic import Anthropic

load_dotenv()
UC = Path("use-case").resolve()
sys.path.insert(0, str(UC))
llm = Anthropic()
MODELO = os.getenv("MODELO", "claude-haiku-5-5")
MODELO_FORTE = os.getenv("MODELO_FORTE", "claude-sonnet-5-5")
subprocess.run([sys.executable, "gerar_documentos.py", "--n", "20", "--notas"], cwd=UC, check=True)

## 1. Ler uma fatura com um modelo de visão

In [ ]:
img = UC / "entrada" / "doc_000.jpg"
display(Mostrar(filename=str(img), width=420))
b64 = base64.standard_b64encode(img.read_bytes()).decode()
bloco_img = {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b64}}
r = llm.messages.create(model=MODELO_FORTE, max_tokens=800, messages=[{"role": "user", "content": [
    bloco_img, {"type": "text", "text": "Que documento é este? Resume em 3 linhas e diz se vês algo de estranho."}]}])
print(r.content[-1].text)
print(f"\nTokens de entrada (a imagem conta): {r.usage.input_tokens}")

In [ ]:
from validacao import Documento, validar
from processar import PROMPT

r = llm.messages.parse(model=MODELO_FORTE, max_tokens=4000, output_format=Documento,
                       messages=[{"role": "user", "content": [bloco_img, {"type": "text", "text": PROMPT}]}])
doc = r.parsed_output
print(doc.model_dump_json(indent=1))

**Exercício 1.1.** Experimenta o mesmo com o `MODELO` (Haiku). Em que campos falha mais? Quanto poupas por página?

## 2. Validar com código

O modelo copia o que está impresso; o código verifica se faz sentido. Procuramos uma fatura com erro inserido e vemos a validação a apanhá-la.

In [ ]:
verdade = [json.loads(l) for l in (UC / "entrada/verdade.jsonl").read_text(encoding="utf-8").splitlines()]
print("Problemas na doc_000:", validar(doc) or "nenhum")
com_erro = next((v for v in verdade if v["erros_inseridos"]), None)
if com_erro:
    print(f"\n{com_erro['ficheiro']} tem um erro inserido: {com_erro['erros_inseridos']}")
    print("Validação dos valores verdadeiros:", validar(Documento(**com_erro["documento"])))

Repara na diferença entre os dois tipos de problema: um erro de leitura do modelo (corrige-se com uma segunda leitura) e um erro no próprio documento (o fornecedor somou mal: tem de ir a uma pessoa). A segunda leitura no `processar.py` diz ao modelo para manter os valores impressos, para não «corrigir» o documento e esconder o erro.

**Exercício 2.1.** Acrescenta ao `validacao.py` uma regra que avise quando o NIF do cliente não é o da Loja Lusa (515123455). Que outras regras de negócio farias?

## 3. Voz: síntese, transcrição e WER

In [ ]:
notas = [v for v in verdade if v["nota"]]
audio = UC / "entrada" / notas[0]["ficheiro"].replace(".jpg", ".wav")
print("Texto original:", notas[0]["nota"])
display(Audio(filename=str(audio)))

In [ ]:
import torch
from transformers import pipeline
modelo_whisper = "openai/whisper-large-v3-turbo" if torch.cuda.is_available() else "openai/whisper-small"
asr = pipeline("automatic-speech-recognition", model=modelo_whisper, device=0 if torch.cuda.is_available() else -1)

def wer(ref: str, hip: str) -> float:
    """Taxa de erro de palavras: distância de edição entre listas de palavras / palavras da referência."""
    norm = lambda s: "".join(c for c in s.lower() if c.isalnum() or c.isspace()).split()
    r, h = norm(ref), norm(hip)
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        ant, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            ant, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, ant + (r[i - 1] != h[j - 1]))
    return d[len(h)] / max(1, len(r))

for n in notas[:4]:
    t0 = time.time()
    txt = asr(str(UC / "entrada" / n["ficheiro"].replace(".jpg", ".wav")), generate_kwargs={"language": "portuguese"})["text"]
    print(f"WER {wer(n['nota'], txt):5.1%} · {time.time() - t0:.1f} s · {txt.strip()}")

A voz sintética é mais limpa do que uma nota real gravada num armazém. Grava uma nota tua com o telemóvel (m4a), põe-na em `entrada/` com o nome de uma fatura e compara.

**Exercício 3.1.** Compara `whisper-small` e `whisper-large-v3-turbo` em WER e tempo. Qual escolhias para 500 notas por dia num servidor sem GPU?

## 4. Raciocínio: esforço e self-consistency

Problemas com resposta numérica conhecida. Comparamos o mesmo modelo com esforço baixo e alto, e um modelo pequeno com voto maioritário de 5 respostas.

In [ ]:
import re
PROBLEMAS = [
    ("Uma loja vende toalhas a 6,90 € sem IVA. Um cliente compra 7 toalhas e tem 10 % de desconto sobre o preço sem IVA. "
     "Com IVA a 23 %, quanto paga, em euros, arredondado ao cêntimo?", 53.47),
    ("Três armazéns enviam 120, 85 e 140 caixas. Cada camião leva no máximo 48 caixas e não mistura armazéns. Quantos camiões são precisos?", 8),
    ("Um preço subiu 20 % em janeiro e desceu 20 % em fevereiro. Era 250 € no início. Quanto custa no fim de fevereiro?", 240),
    ("A Ana faz um relatório em 6 horas e o Rui em 3 horas. Trabalhando juntos ao mesmo ritmo, quantos minutos demoram?", 120),
]

def numero(txt):
    nums = re.findall(r"-?\d+(?:[.,]\d+)?", txt.replace(" ", ""))
    return float(nums[-1].replace(",", ".")) if nums else None

def perguntar(modelo, p, esforco=None):
    kw = {"output_config": {"effort": esforco}} if esforco else {}
    r = llm.messages.create(model=modelo, max_tokens=8000, **kw, messages=[{"role": "user", "content":
        p + "\nTermina com uma linha 'Resposta: <número>'."}])
    return numero(r.content[-1].text), r.usage.output_tokens

for nome, f in [("Sonnet esforço baixo", lambda p: perguntar(MODELO_FORTE, p, "low")),
                ("Sonnet esforço alto", lambda p: perguntar(MODELO_FORTE, p, "high"))]:
    certos, toks = 0, 0
    for p, resp in PROBLEMAS:
        v, t = f(p)
        certos += v is not None and abs(v - resp) < 0.01
        toks += t
    print(f"{nome:<22} {certos}/{len(PROBLEMAS)} certos · {toks} tokens de saída")

certos, toks = 0, 0
for p, resp in PROBLEMAS:
    votos = [perguntar(MODELO, p) for _ in range(5)]
    v = Counter(x for x, _ in votos if x is not None).most_common(1)[0][0]
    certos += abs(v - resp) < 0.01
    toks += sum(t for _, t in votos)
print(f"{'Haiku × 5 com voto':<22} {certos}/{len(PROBLEMAS)} certos · {toks} tokens de saída")

Converte tokens em custo com os preços de cada modelo. Para problemas destes, o modelo grande com pouco esforço muitas vezes basta; em problemas mais difíceis, a diferença aparece.

**Exercício 4.1.** Escreve 5 problemas mais difíceis (por exemplo, de planeamento com restrições) e repete. A partir de que dificuldade compensa o esforço alto?

## 5. Um modelo pequeno no teu computador (opcional)

Com o Ollama instalado (`ollama pull qwen2.5:3b`), o mesmo cliente compatível com a API da OpenAI fala com o modelo local:

In [ ]:
try:
    from openai import OpenAI
    local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    t0 = time.time()
    r = local.chat.completions.create(model="qwen2.5:3b", messages=[{"role": "user", "content":
        "Classifica numa palavra (entrega, produto, faturação, outro): 'Cobraram-me duas vezes a mesma encomenda.'"}])
    print(r.choices[0].message.content, f"({time.time() - t0:.1f} s, sem sair do computador)")
except Exception as e:
    print("Ollama não disponível:", type(e).__name__)

## 6. O pipeline completo

In [ ]:
p = subprocess.run([sys.executable, "processar.py", "--whisper", modelo_whisper], cwd=UC, capture_output=True, text=True)
print(p.stdout[-3000:], p.stderr[-1500:] if p.returncode else "")
p = subprocess.run([sys.executable, "avaliar.py"], cwd=UC, capture_output=True, text=True)
print(p.stdout)

In [ ]:
import pandas as pd, sqlite3
with sqlite3.connect(UC / "documentos.db") as con:
    display(pd.read_sql("SELECT ficheiro, emitente, total, estado, problemas, centro_custo, alerta FROM documentos", con))

O número que importa ao negócio é a última linha do `avaliar.py`: documentos aceites automaticamente com algum campo errado. Os que vão para revisão custam tempo; os que passam errados custam dinheiro.

**Exercício 6.1.** Gera 30 documentos com `--seed 99` e mais ruído (aumenta o desfoque em `desenhar`). Como mudam a exatidão e a taxa de revisão?

## 7. AI Act: classificar os teus projetos

Para cada use case do curso, decide o nível de risco e as obrigações. Compara com a tua intuição e com a demo da lição.

In [ ]:
projetos = {
    "M03 assistente de apoio ao cliente (Gradio)": "",
    "M06 assistente de legislação fiscal": "",
    "M10 equipa de pesquisa de mercado": "",
    "M11 classificador de reclamações": "",
    "M13 processador de faturas": "",
    "Variante: o classificador decide sozinho reembolsos acima de 500 €": "",
}
# Preenche com: proibido / alto / transparência / mínimo, e uma frase de justificação.
for k, v in projetos.items():
    print(f"- {k}: {v or '?'}")

## Use case: processador de documentos digitalizados

```bash
cd modulos/13-multimodal-e-fronteira-do-estado-da-arte/use-case
uv run python gerar_documentos.py --n 30 --notas
uv run python processar.py
uv run python avaliar.py
```

**Desafio:** troca o `extrair()` por um VLM aberto local (`Qwen/Qwen2.5-VL-7B-Instruct` com transformers ou vLLM, pedindo JSON) e compara exatidão, tempo e custo com o Claude. Em que ponto o modelo local é bom o suficiente para documentos que não podem sair da empresa?